# 08. Lecke – Többügynökös tervezési minta


## Beállítás


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Miért Multi-agent rendszerek?

A való életbeli feladatok, mint például az utazástervezés, sokféle szakértelmet igényelnek — logisztika, helyi ismeretek, költségvetés és még sok más. Egyetlen ügynök, aki mindent próbál kezelni, gyorsan átláthatatlanná válik.

A multi-agent rendszerek ezt **specializációval** oldják meg: minden ügynök egy adott szakterületre fókuszál, így magasabb minőségű eredményeket produkál, mint egy általános szakember. Emellett javítják a **skálázhatóságot** is — új ügynököket adhatunk hozzá (pl. egy repülőjegy szakértőt, egy étteremkritikust) anélkül, hogy újra kellene írni a meglévő munkafolyamatot. Az ügynökök egy strukturált csövön keresztül állnak össze, átadva egymásnak a kontextust.


## Speciális ügynökök létrehozása


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Egymást követő munkafolyamat létrehozása

A `WorkflowBuilder` lehetővé teszi, hogy ágenseket egy irányított gráfba köss össze. Itt létrehozunk egy egyszerű, kétszakaszos munkafolyamatot: a **TravelPlanner** megtervezi az útitervet, majd a **TravelConcierge** átnézi és kibővíti azt.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## További ügynökök hozzáadása a munkafolyamathoz

A többügynökös minta egyik legnagyobb előnye az egyszerű bővíthetőség. Lent hozzáadunk egy **BudgetReviewer** ügynököt, amely átnézi a tervet az utazó költségvetése alapján, jelöli azokat a tételeket, amelyek meghaladhatják a költségkeretet, és pénzmegtakarítást javasló alternatívákat kínál. A munkafolyamat most három ügynököt futtat egymás után:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Készíts szakosodott ügynököket** — mindegyiknek egy fókuszált szerepe van (tervezés, concierge, költségvetés áttekintése).
2. **Kösd össze az ügynököket egy szekvenciális munkafolyamatba** a `WorkflowBuilder` és az `add_edge` használatával.
3. **Folymatosan közvetítsd a kimenetet** egy több ügynökös csővezetéken, nyomon követve, melyik ügynök beszél.
4. **Bővíts egy munkafolyamatot** új ügynökök hozzáadásával a lánchoz anélkül, hogy a meglévőket módosítanád.

A multi-ügynök tervezési minta egyszerűvé teszi az egyes ügynököket, miközben gazdagabb, alaposabb, egymás által átnézett eredményeket állít elő, mint amit egyetlen ügynök önmagában elérhetne.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
